In [5]:
# Create a shell script that uses the commandline gdal libraries to clip 
# a series of orthoimages as GEOTIFFs.
#
# Usage: run the generated shell script on a machine with the GDAL/OGR libraries

from osgeo import gdal
from osgeo import ogr
import geopandas as gpd
from shapely import wkt, box
import rasterio
from rasterio.mask import mask
import json
import csv
import glob

PATH = "../download" # main data directory
SRC = "orthophotos" # src directory for drone survey orthoimages
DEST = "ortho_clipped" # destination for clipped images

plots = ogr.Open(f"{PATH}/tree_plots_polygons.geojson")
layer = plots.GetLayerByIndex(0)

rasters = glob.glob(f"{PATH}/{SRC}/*.tif")
rasters.sort()

# step through original orthoimages from drone survey
for raster in rasters:
    with rasterio.open(raster) as src:
        # get bounds of original orthoimage
        bounds = src.bounds
        crs = src.crs
        raster_bounds = ogr.CreateGeometryFromWkb(box(*bounds).wkb)

        # step through individual survey polygons
        for feature in layer:
        
            # check to see if survey polygon is in orthoimage bounds
            geometry = feature.GetGeometryRef()
            is_within = geometry.Within(raster_bounds)
            if is_within: 

                # create a geometry from the survey area and a 30 meter buffer
                gdf = gpd.GeoDataFrame([1], geometry=[wkt.loads(geometry.ExportToWkt())], crs='epsg:32617')
                clip_rect = gdf.geometry.buffer(30, cap_style='flat', join_style='mitre').to_wkt()[0]
                
                # write the buffer file for use in the shell script
                cutfile = f"{PATH}/{DEST}/{feature['name']}.csv"
                with open(cutfile, 'w', newline='') as csvfile:
                    csv_writer = csv.writer(csvfile)
                    data = [
                        ['id','WKT'],
                        [1,clip_rect]
                    ]
                    for row in data:
                        csv_writer.writerow(row)
                        
                # print a shell script to use the buffer geometry to clip the orthoimage
                sh = (
                    f"gdalwarp -cutline {cutfile} "
                    f"-crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 "
                    f"'{raster}' {PATH}/{DEST}/{feature['name']}.tif"
                )
                print(sh)

gdalwarp -cutline ../download/ortho_clipped/plot_13_1.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\2025BICY_20250116_b14f1_2_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_13_1.tif
gdalwarp -cutline ../download/ortho_clipped/plot_13_2.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\2025BICY_20250116_b14f1_2_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_13_2.tif
gdalwarp -cutline ../download/ortho_clipped/plot_15_2.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\2025BICY_20250116_b14f1_2_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_15_2.tif
gdalwarp -cutline ../download/ortho_clipped/plot_15_3.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\2025BICY_20250116_b14f1_2_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_15_3.tif
gdalwarp -cutline ../download/ortho_clipped/plot_3_1.csv -crop_to_cu

gdalwarp -cutline ../download/ortho_clipped/plot_12_3.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\BICY2025_RINEX20250219_b8_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_12_3.tif
gdalwarp -cutline ../download/ortho_clipped/plot_9_1.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\BICY_20250206_b6_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_9_1.tif
gdalwarp -cutline ../download/ortho_clipped/plot_9_2.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\BICY_20250206_b6_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_9_2.tif
gdalwarp -cutline ../download/ortho_clipped/plot_9_3.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs EPSG:32617 '../download/orthophotos\BICY_20250206_b6_transparent_mosaic_group1.tif' ../download/ortho_clipped/plot_9_3.tif
gdalwarp -cutline ../download/ortho_clipped/plot_10_3.csv -crop_to_cutline -s_srs EPSG:32617 -t_srs E